In [108]:
import pandas as pd

In [ ]:
team_stats = pd.read_csv("../../data/foot/whoScored.csv")

In [ ]:
games = pd.read_csv("../../data/foot/dataset_final.csv")

In [111]:
team_stats.head()

,Équipe,Domicile_General_Buts,Domicile_General_Tirs pm,Domicile_General_Possession%,Domicile_General_PassesRéussies%,Domicile_General_AériensGagnés,Domicile_General_Note,Domicile_General_Cartons_J,Domicile_General_Cartons_R,Exterieur_General_Buts,...,Exterieur_Detailed_Saves_Note,Exterieur_Detailed_Possession-loss_ControlesRatés,Exterieur_Detailed_Possession-loss_Dépossédé,Exterieur_Detailed_Possession-loss_Note,Exterieur_Detailed_Aerial_Total,Exterieur_Detailed_Aerial_Gagnés,Exterieur_Detailed_Aerial_Perdus,Exterieur_Detailed_Aerial_Note,Championnat,Saison
0,Brest,25,15.9,0.564,0.825,19.4,6.79,40,1,28,...,6.72,16.5,7.7,6.72,31.6,16.1,15.5,6.72,Ligue_1,2023-2024
1,Clermont Foot,14,12.9,0.495,0.818,11.5,6.50,25,4,12,...,6.44,14.8,8.1,6.44,24.9,11.9,12.9,6.44,Ligue_1,2023-2024
2,Le Havre,18,13.4,0.455,0.803,14.4,6.60,30,1,16,...,6.48,13.6,8.1,6.48,30.2,14.9,15.4,6.48,Ligue_1,2023-2024
3,Lens,27,16.6,0.523,0.839,14.5,6.68,28,2,18,...,6.58,14.7,8.5,6.58,24.9,12.8,12.1,6.58,Ligue_1,2023-2024
4,Lille,34,14.0,0.558,0.853,11.6,6.84,30,2,18,...,6.63,14.3,8.8,6.63,20.1,10.1,10.0,6.63,Ligue_1,2023-2024


In [112]:
missing_stats = team_stats.isnull().sum()

# Ne garder que les colonnes qui ont au moins une valeur manquante
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
missing_stats.head()

Domicile_xG_Pour_xG        14
Domicile_xG_Pour_Buts*     14
Domicile_xG_Pour_xGDiff    14
Domicile_xG_Pour_Tirs      14
Domicile_xG_Pour_xG/Tir    14
dtype: int64

In [113]:
def impute_custom_means(df, columns_to_fix):
    for col in columns_to_fix:
        # 1. Calcul des moyennes par année (Ligue)
        year_means = df.groupby('Saison')[col].transform('mean')
        
        # 2. Calcul des moyennes par équipe (Historique)
        team_means = df.groupby('Équipe')[col].transform('mean')
        
        # 3. Calcul de la moyenne hybride
        hybrid_mean = (year_means + team_means) / 2
        
        # 4. Remplissage sélectif
        # Si une équipe est nouvelle (pas d'historique), on peut fallback sur la moyenne de l'année
        fill_values = hybrid_mean.fillna(year_means)
        
        df[col] = df[col].fillna(fill_values)
    
    return df

# Liste des colonnes avec des valeurs manquantes (basée sur votre série missing_stats)
cols_to_impute = missing_stats.index.tolist()

# Application
team_stats = impute_custom_means(team_stats, cols_to_impute)

In [114]:
missing_stats = team_stats.isnull().sum()

# Ne garder que les colonnes qui ont au moins une valeur manquante
missing_stats = missing_stats[missing_stats > 0].sort_values(ascending=False)
missing_stats.head()

Series([], dtype: int64)

In [115]:
games.head()

,League,Season,Date,Time,Home,Score,Away,Attendance,Venue,Referee,...,clean_sheet_pct_home,fail_to_score_pct_home,preferred_formation_away,profile_away,team_style_away,pressing_intensity_away,defensive_line_away,win_pct_away,clean_sheet_pct_away,fail_to_score_pct_away
0,La Liga,2025-2026,2025-08-15,19:00,Girona,1–3,Rayo Vallecano,12403.0,Estadi Municipal de Montilivi,Javier Alberola Rojas,...,20.41,23.13,4-2-3-1,balanced,hard,0.726109,mid,29.70,27.00,35.10
1,La Liga,2025-2026,2025-08-15,21:30,Villarreal,2–0,Oviedo,18333.0,Estadio de la Cerámica,Alejandro Muñiz Ruiz,...,22.70,25.00,4-4-2,attacking,possession,0.750000,high,33.33,33.33,66.67
2,La Liga,2025-2026,2025-08-16,19:30,Mallorca,0–3,Barcelona,23318.0,Estadi Mallorca Son Moix,José Luis Munuera Montero,...,26.63,34.32,4-2-3-1,attacking,possession,0.860000,high,77.78,41.11,3.33
3,La Liga,2025-2026,2025-08-16,21:30,Alavés,2–1,Levante,12837.0,Estadio de Mendizorroza,Miguel Sesma Espinosa,...,0.00,0.00,4-2-3-1,attacking,gegenpress,0.870000,high,14.29,14.29,28.57
4,La Liga,2025-2026,2025-08-16,21:30,Valencia,1–1,Real Sociedad,45333.0,Estadio de Mestalla,Jose Maria Sanchez Santos,...,28.90,21.10,3-4-3,defensive,counter,0.320000,low,27.03,10.81,37.84


In [116]:
len(team_stats["Équipe"].unique().tolist())

130

In [117]:
len(games["Home"].unique().tolist())

130

In [118]:
games[~games["Home"].isin(team_stats["Équipe"].unique())]["Home"].unique()

<ArrowStringArray>
[           'Alavés',   'Atlético Madrid',            'Oviedo',
        'Valladolid',           'Leganés',           'Almería',
             'Cádiz',        'Heidenheim',        'Leverkusen',
          'St Pauli',          'Gladbach',         'Stuttgart',
          'Dortmund',              'Köln',      'Darmstadt 98',
        'Hertha BSC',           'Arminia',    'Greuther Fürth',
 'Tottenham Hotspur',    'Manchester Utd',      'Leeds United',
   'West Ham United',  'Newcastle United',      'Ipswich Town',
    'Leicester City',        'Luton Town',      'Norwich City',
     'Saint-Étienne',           'Ajaccio',             'Milan',
             'Parma',     'Hellas Verona']
Length: 32, dtype: str

In [119]:
team_mapping = {
    # France
    'Brest': 'Brest',
    'Clermont Foot': 'Clermont Foot',
    'Le Havre': 'Le Havre',
    'Lens': 'Lens',
    'Lille': 'Lille',
    'Lorient': 'Lorient',
    'Lyon': 'Lyon',
    'Marseille': 'Marseille',
    'Metz': 'Metz',
    'Monaco': 'Monaco',
    'Montpellier': 'Montpellier',
    'Nantes': 'Nantes',
    'Nice': 'Nice',
    'Paris Saint-Germain': 'Paris Saint-Germain',
    'Reims': 'Reims',
    'Rennes': 'Rennes',
    'Strasbourg': 'Strasbourg',
    'Toulouse': 'Toulouse',
    'Angers': 'Angers',
    'Auxerre': 'Auxerre',
    'Saint-Etienne': 'Saint-Étienne',
    'Paris FC': 'Paris FC',
    'AC Ajaccio': 'Ajaccio',
    'Troyes': 'Troyes',
    'Bordeaux': 'Bordeaux',

    # Angleterre
    'Arsenal': 'Arsenal',
    'Aston Villa': 'Aston Villa',
    'Bournemouth': 'Bournemouth',
    'Brentford': 'Brentford',
    'Brighton': 'Brighton',
    'Burnley': 'Burnley',
    'Chelsea': 'Chelsea',
    'Crystal Palace': 'Crystal Palace',
    'Everton': 'Everton',
    'Fulham': 'Fulham',
    'Liverpool': 'Liverpool',
    'Luton': 'Luton Town',
    'Manchester City': 'Manchester City',
    'Manchester United': 'Manchester Utd',
    'Newcastle': 'Newcastle United',
    'Nottingham Forest': 'Nottingham Forest',
    'Sheffield United': 'Sheffield United',
    'Tottenham': 'Tottenham Hotspur',
    'West Ham': 'West Ham United',
    'Wolves': 'Wolves',
    'Ipswich': 'Ipswich Town',
    'Leicester': 'Leicester City',
    'Southampton': 'Southampton',
    'Leeds': 'Leeds United',
    'Sunderland': 'Sunderland',
    'Norwich': 'Norwich City',
    'Watford': 'Watford',

    # Allemagne
    'Augsburg': 'Augsburg',
    'Bayer Leverkusen': 'Leverkusen',
    'Bayern Munich': 'Bayern Munich',
    'Bochum': 'Bochum',
    'Borussia Dortmund': 'Dortmund',
    'Borussia M.Gladbach': 'Gladbach',
    'Darmstadt': 'Darmstadt 98',
    'Eintracht Frankfurt': 'Eintracht Frankfurt',
    'FC Heidenheim': 'Heidenheim',
    'FC Koln': 'Köln',
    'Freiburg': 'Freiburg',
    'Hoffenheim': 'Hoffenheim',
    'Mainz 05': 'Mainz 05',
    'RB Leipzig': 'RB Leipzig',
    'Union Berlin': 'Union Berlin',
    'VfB Stuttgart': 'Stuttgart',
    'Werder Bremen': 'Werder Bremen',
    'Wolfsburg': 'Wolfsburg',
    'Holstein Kiel': 'Holstein Kiel',
    'St. Pauli': 'St Pauli',
    'Hamburger SV': 'Hamburger SV',
    'Hertha Berlin': 'Hertha BSC',
    'Schalke 04': 'Schalke 04',
    'Arminia Bielefeld': 'Arminia',
    'Greuther Fuerth': 'Greuther Fürth',

    # Espagne
    'Almeria': 'Almería',
    'Athletic Club': 'Athletic Club',
    'Atletico Madrid': 'Atlético Madrid',
    'Barcelona': 'Barcelona',
    'Cadiz': 'Cádiz',
    'Celta Vigo': 'Celta Vigo',
    'Deportivo Alaves': 'Alavés',
    'Getafe': 'Getafe',
    'Girona': 'Girona',
    'Granada': 'Granada',
    'Las Palmas': 'Las Palmas',
    'Mallorca': 'Mallorca',
    'Osasuna': 'Osasuna',
    'Rayo Vallecano': 'Rayo Vallecano',
    'Real Betis': 'Real Betis',
    'Real Madrid': 'Real Madrid',
    'Real Sociedad': 'Real Sociedad',
    'Sevilla': 'Sevilla',
    'Valencia': 'Valencia',
    'Villarreal': 'Villarreal',
    'Espanyol': 'Espanyol',
    'Leganes': 'Leganés',
    'Real Valladolid': 'Valladolid',
    'Elche': 'Elche',
    'Levante': 'Levante',
    'Real Oviedo': 'Oviedo',

    # Italie
    'AC Milan': 'Milan',
    'Atalanta': 'Atalanta',
    'Bologna': 'Bologna',
    'Cagliari': 'Cagliari',
    'Empoli': 'Empoli',
    'Fiorentina': 'Fiorentina',
    'Frosinone': 'Frosinone',
    'Genoa': 'Genoa',
    'Inter': 'Inter',
    'Juventus': 'Juventus',
    'Lazio': 'Lazio',
    'Lecce': 'Lecce',
    'Monza': 'Monza',
    'Napoli': 'Napoli',
    'Roma': 'Roma',
    'Salernitana': 'Salernitana',
    'Sassuolo': 'Sassuolo',
    'Torino': 'Torino',
    'Udinese': 'Udinese',
    'Verona': 'Hellas Verona',
    'Como': 'Como',
    'Parma Calcio 1913': 'Parma',
    'Venezia': 'Venezia',
    'Cremonese': 'Cremonese',
    'Pisa': 'Pisa',
    'Sampdoria': 'Sampdoria',
    'Spezia': 'Spezia'
}

In [120]:
team_stats["Équipe"] = team_stats["Équipe"].map(team_mapping)

In [121]:
team_stats.head()

,Équipe,Domicile_General_Buts,Domicile_General_Tirs pm,Domicile_General_Possession%,Domicile_General_PassesRéussies%,Domicile_General_AériensGagnés,Domicile_General_Note,Domicile_General_Cartons_J,Domicile_General_Cartons_R,Exterieur_General_Buts,...,Exterieur_Detailed_Saves_Note,Exterieur_Detailed_Possession-loss_ControlesRatés,Exterieur_Detailed_Possession-loss_Dépossédé,Exterieur_Detailed_Possession-loss_Note,Exterieur_Detailed_Aerial_Total,Exterieur_Detailed_Aerial_Gagnés,Exterieur_Detailed_Aerial_Perdus,Exterieur_Detailed_Aerial_Note,Championnat,Saison
0,Brest,25,15.9,0.564,0.825,19.4,6.79,40,1,28,...,6.72,16.5,7.7,6.72,31.6,16.1,15.5,6.72,Ligue_1,2023-2024
1,Clermont Foot,14,12.9,0.495,0.818,11.5,6.50,25,4,12,...,6.44,14.8,8.1,6.44,24.9,11.9,12.9,6.44,Ligue_1,2023-2024
2,Le Havre,18,13.4,0.455,0.803,14.4,6.60,30,1,16,...,6.48,13.6,8.1,6.48,30.2,14.9,15.4,6.48,Ligue_1,2023-2024
3,Lens,27,16.6,0.523,0.839,14.5,6.68,28,2,18,...,6.58,14.7,8.5,6.58,24.9,12.8,12.1,6.58,Ligue_1,2023-2024
4,Lille,34,14.0,0.558,0.853,11.6,6.84,30,2,18,...,6.63,14.3,8.8,6.63,20.1,10.1,10.0,6.63,Ligue_1,2023-2024


In [122]:
games.head()

,League,Season,Date,Time,Home,Score,Away,Attendance,Venue,Referee,...,clean_sheet_pct_home,fail_to_score_pct_home,preferred_formation_away,profile_away,team_style_away,pressing_intensity_away,defensive_line_away,win_pct_away,clean_sheet_pct_away,fail_to_score_pct_away
0,La Liga,2025-2026,2025-08-15,19:00,Girona,1–3,Rayo Vallecano,12403.0,Estadi Municipal de Montilivi,Javier Alberola Rojas,...,20.41,23.13,4-2-3-1,balanced,hard,0.726109,mid,29.70,27.00,35.10
1,La Liga,2025-2026,2025-08-15,21:30,Villarreal,2–0,Oviedo,18333.0,Estadio de la Cerámica,Alejandro Muñiz Ruiz,...,22.70,25.00,4-4-2,attacking,possession,0.750000,high,33.33,33.33,66.67
2,La Liga,2025-2026,2025-08-16,19:30,Mallorca,0–3,Barcelona,23318.0,Estadi Mallorca Son Moix,José Luis Munuera Montero,...,26.63,34.32,4-2-3-1,attacking,possession,0.860000,high,77.78,41.11,3.33
3,La Liga,2025-2026,2025-08-16,21:30,Alavés,2–1,Levante,12837.0,Estadio de Mendizorroza,Miguel Sesma Espinosa,...,0.00,0.00,4-2-3-1,attacking,gegenpress,0.870000,high,14.29,14.29,28.57
4,La Liga,2025-2026,2025-08-16,21:30,Valencia,1–1,Real Sociedad,45333.0,Estadio de Mestalla,Jose Maria Sanchez Santos,...,28.90,21.10,3-4-3,defensive,counter,0.320000,low,27.03,10.81,37.84


In [123]:
home_cols_to_keep = [col for col in team_stats.columns if col.startswith('Domicile_')]
team_home_stats = team_stats[['Équipe', 'Saison'] + home_cols_to_keep]

away_cols_to_keep = [col for col in team_stats.columns if col.startswith('Exterieur_')]
team_away_stats = team_stats[['Équipe', 'Saison'] + away_cols_to_keep]

df_final = games.merge(
    team_home_stats,
    left_on=["Home", "Season"], right_on=['Équipe', 'Saison'], how='left'
).rename(columns={col: f"{col}_home" for col in team_home_stats.columns if (col not in ['Équipe','Saison'])}
).drop(columns=['Équipe', 'Saison'], errors='ignore')

df_final = df_final.merge(
    team_away_stats,
    left_on=["Away", "Season"], right_on=['Équipe', 'Saison'], how='left'
).rename(columns={col: f"{col}_away" for col in team_away_stats.columns if (col not in ['Équipe','Saison'])}
).drop(columns=['Équipe', 'Saison'], errors='ignore')

In [124]:
df_final.head()

,League,Season,Date,Time,Home,Score,Away,Attendance,Venue,Referee,...,Exterieur_Detailed_Saves_SurfaceReparation_away,Exterieur_Detailed_Saves_EnDehorsSurface_away,Exterieur_Detailed_Saves_Note_away,Exterieur_Detailed_Possession-loss_ControlesRatés_away,Exterieur_Detailed_Possession-loss_Dépossédé_away,Exterieur_Detailed_Possession-loss_Note_away,Exterieur_Detailed_Aerial_Total_away,Exterieur_Detailed_Aerial_Gagnés_away,Exterieur_Detailed_Aerial_Perdus_away,Exterieur_Detailed_Aerial_Note_away
0,La Liga,2025-2026,2025-08-15,19:00,Girona,1–3,Rayo Vallecano,12403.0,Estadi Municipal de Montilivi,Javier Alberola Rojas,...,1.6,0.8,6.44,17.5,8.5,6.44,21.6,8.9,12.7,6.44
1,La Liga,2025-2026,2025-08-15,21:30,Villarreal,2–0,Oviedo,18333.0,Estadio de la Cerámica,Alejandro Muñiz Ruiz,...,2.8,1.6,6.41,16.2,7.7,6.41,25.2,12.4,12.8,6.41
2,La Liga,2025-2026,2025-08-16,19:30,Mallorca,0–3,Barcelona,23318.0,Estadi Mallorca Son Moix,José Luis Munuera Montero,...,2.4,0.6,6.85,14.2,9.2,6.85,19.6,12.0,7.6,6.85
3,La Liga,2025-2026,2025-08-16,21:30,Alavés,2–1,Levante,12837.0,Estadio de Mendizorroza,Miguel Sesma Espinosa,...,2.9,0.8,6.48,15.8,7.6,6.48,27.2,13.2,14.1,6.48
4,La Liga,2025-2026,2025-08-16,21:30,Valencia,1–1,Real Sociedad,45333.0,Estadio de Mestalla,Jose Maria Sanchez Santos,...,1.8,1.0,6.55,15.1,7.4,6.55,26.9,13.8,13.1,6.55


In [125]:
df_final.shape

(8681, 308)

In [ ]:
df_final.to_csv("../../data/foot/processed.csv")